In [1]:
import pandas as pd
from pathlib import Path
import gc

In [4]:
# ============================================================
# CONFIGURATION
# ============================================================

def find_project_root():
    """
    Find project root containing data/tables.
    Works whether notebook is launched from root or notebooks/.
    """
    current = Path.cwd().resolve()

    for path in [current, *current.parents]:
        if (path / "data" / "tables").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root containing data/tables/"
    )


PROJECT_ROOT = find_project_root()

TABLES_DIR = PROJECT_ROOT / "data" / "tables"
PART1_DIR = TABLES_DIR / "part_1"
ABS_DIR = PROJECT_ROOT / "data" / "raw_abs"

CURATED_DIR = (
    PROJECT_ROOT
    / "data"
    / "curated"
    / "curated_transactions"
)

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Temporary/provisional fraud threshold
FRAUD_THRESHOLD = 80


print("PROJECT ROOT:", PROJECT_ROOT)
print("TABLES:", TABLES_DIR)
print("RAW ABS:", ABS_DIR)
print("CURATED OUTPUT:", CURATED_DIR)

PROJECT ROOT: /Users/amyliu/Desktop/MAST30034_Project_2
TABLES: /Users/amyliu/Desktop/MAST30034_Project_2/data/tables
RAW ABS: /Users/amyliu/Desktop/MAST30034_Project_2/data/raw_abs
CURATED OUTPUT: /Users/amyliu/Desktop/MAST30034_Project_2/data/curated/curated_transactions


In [5]:
# ============================================================
# PART 1 — LOAD
# ============================================================

def load_part1():

    consumer = pd.read_csv(
        PART1_DIR / "tbl_consumer.csv",
        sep="|"
    )

    consumer_details = pd.read_parquet(
        PART1_DIR / "consumer_user_details.parquet"
    )

    merchants = pd.read_parquet(
        PART1_DIR / "tbl_merchants.parquet"
    )

    consumer_fraud = pd.read_csv(
        PART1_DIR / "consumer_fraud_probability.csv"
    )

    merchant_fraud = pd.read_csv(
        PART1_DIR / "merchant_fraud_probability.csv"
    )

    return (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    )

In [6]:
# ============================================================
# PART 1 — CLEAN
# ============================================================

def clean_consumer(df):

    df = df.copy()

    # Postcodes are identifiers, not measurements
    df["postcode"] = (
        pd.to_numeric(
            df["postcode"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(4)
    )

    return df


def clean_consumer_fraud(df):

    df = df.copy()

    # Remove exact duplicates
    df = df.drop_duplicates()

    # Fraud data is day-level
    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        dayfirst=True,
        errors="raise"
    ).dt.normalize()

    df["fraud_probability"] = pd.to_numeric(
        df["fraud_probability"],
        errors="raise"
    )

    return df


def clean_merchant_fraud(df):

    df = df.copy()

    df = df.drop_duplicates()

    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        dayfirst=True,
        errors="raise"
    ).dt.normalize()

    df["fraud_probability"] = pd.to_numeric(
        df["fraud_probability"],
        errors="raise"
    )

    return df

In [7]:
# ============================================================
# VALIDATION
# ============================================================

VALID_STATES = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

VALID_GENDERS = {
    "Male", "Female", "Undisclosed"
}


def validate_consumer(df):

    assert df["consumer_id"].notna().all()
    assert df["consumer_id"].is_unique

    assert df["state"].isin(VALID_STATES).all()
    assert df["gender"].isin(VALID_GENDERS).all()

    assert df["postcode"].str.len().eq(4).all()

    assert df["name"].str.strip().ne("").all()
    assert df["address"].str.strip().ne("").all()


def validate_consumer_details(df):

    assert df["consumer_id"].notna().all()
    assert df["user_id"].notna().all()

    assert df["consumer_id"].is_unique
    assert df["user_id"].is_unique


def validate_merchants(df):

    assert df.index.notna().all()
    assert df.index.is_unique

    assert (
        pd.Series(df.index.astype(str))
        .str.len()
        .eq(11)
        .all()
    )

    assert df["name"].notna().all()
    assert df["name"].str.strip().ne("").all()


def validate_consumer_fraud(df):

    assert df["user_id"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert df.duplicated().sum() == 0


def validate_merchant_fraud(df):

    assert df["merchant_abn"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert pd.api.types.is_datetime64_any_dtype(
        df["order_datetime"]
    )


def validate_consumer_relationship(
    consumer,
    consumer_details
):

    consumer.merge(
        consumer_details,
        on="consumer_id",
        validate="one_to_one"
    )

    assert (
        set(consumer["consumer_id"])
        ==
        set(consumer_details["consumer_id"])
    )

In [8]:
# ============================================================
# VALIDATION
# ============================================================

VALID_STATES = {
    "ACT", "NSW", "NT", "QLD",
    "SA", "TAS", "VIC", "WA"
}

VALID_GENDERS = {
    "Male", "Female", "Undisclosed"
}


def validate_consumer(df):

    assert df["consumer_id"].notna().all()
    assert df["consumer_id"].is_unique

    assert df["state"].isin(VALID_STATES).all()
    assert df["gender"].isin(VALID_GENDERS).all()

    assert df["postcode"].str.len().eq(4).all()

    assert df["name"].str.strip().ne("").all()
    assert df["address"].str.strip().ne("").all()


def validate_consumer_details(df):

    assert df["consumer_id"].notna().all()
    assert df["user_id"].notna().all()

    assert df["consumer_id"].is_unique
    assert df["user_id"].is_unique


def validate_merchants(df):

    assert df.index.notna().all()
    assert df.index.is_unique

    assert (
        pd.Series(df.index.astype(str))
        .str.len()
        .eq(11)
        .all()
    )

    assert df["name"].notna().all()
    assert df["name"].str.strip().ne("").all()


def validate_consumer_fraud(df):

    assert df["user_id"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert df.duplicated().sum() == 0


def validate_merchant_fraud(df):

    assert df["merchant_abn"].notna().all()
    assert df["order_datetime"].notna().all()
    assert df["fraud_probability"].notna().all()

    assert df["fraud_probability"].between(
        0, 100
    ).all()

    assert pd.api.types.is_datetime64_any_dtype(
        df["order_datetime"]
    )


def validate_consumer_relationship(
    consumer,
    consumer_details
):

    consumer.merge(
        consumer_details,
        on="consumer_id",
        validate="one_to_one"
    )

    assert (
        set(consumer["consumer_id"])
        ==
        set(consumer_details["consumer_id"])
    )

In [9]:
# ============================================================
# EXTERNAL DATA — POSTCODE -> SA2
# ============================================================

def load_sa2_mapping():

    path = ABS_DIR / "poa_to_sa2.csv"

    poa_to_sa2 = pd.read_csv(path)

    poa_to_sa2["POA_CODE_2021"] = (
        pd.to_numeric(
            poa_to_sa2["POA_CODE_2021"],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(4)
    )

    # Choose highest-ratio SA2 for each postcode
    poa_to_sa2_best = (
        poa_to_sa2
        .sort_values(
            "ratio",
            ascending=False
        )
        .drop_duplicates(
            subset="POA_CODE_2021",
            keep="first"
        )
    )

    return poa_to_sa2_best

In [10]:
# ============================================================
# EXTERNAL DATA — SEIFA
# ============================================================

def load_seifa():

    seifa_path = (
        ABS_DIR
        / "SEIFA_2021_SA2.xlsx"
    )

    seifa = pd.read_excel(
        seifa_path,
        sheet_name="Table 1",
        skiprows=6,
        header=None
    )

    seifa.columns = [
        "SA2_CODE_2021",
        "SA2_NAME_2021",
        "disadvantage_score",
        "disadvantage_decile",
        "adv_disadv_score",
        "adv_disadv_decile",
        "economic_resources_score",
        "economic_resources_decile",
        "education_occupation_score",
        "education_occupation_decile",
        "usual_resident_population"
    ]

    # SA2 identifier
    seifa["SA2_CODE_2021"] = pd.to_numeric(
        seifa["SA2_CODE_2021"],
        errors="coerce"
    ).astype("Int64")

    # Remove footer/header rows
    seifa = seifa[
        seifa["SA2_CODE_2021"].notna()
    ].copy()

    # --------------------------------------------------------
    # IMPORTANT PARQUET FIX
    # Convert numeric SEIFA columns explicitly.
    # Values such as "-" become NaN.
    # --------------------------------------------------------

    numeric_cols = [
        "disadvantage_score",
        "disadvantage_decile",
        "adv_disadv_score",
        "adv_disadv_decile",
        "economic_resources_score",
        "economic_resources_decile",
        "education_occupation_score",
        "education_occupation_decile",
        "usual_resident_population"
    ]

    for col in numeric_cols:

        seifa[col] = pd.to_numeric(
            seifa[col],
            errors="coerce"
        )

    # Explicit string column
    seifa["SA2_NAME_2021"] = (
        seifa["SA2_NAME_2021"]
        .astype("string")
    )

    return seifa

In [11]:
# ============================================================
# FRAUD JOINS
# ============================================================

def add_consumer_fraud(
    transactions,
    consumer_fraud
):

    fraud = consumer_fraud[
        [
            "user_id",
            "order_datetime",
            "fraud_probability"
        ]
    ].rename(
        columns={
            "fraud_probability":
            "consumer_fraud_probability"
        }
    )

    return transactions.merge(
        fraud,
        on=[
            "user_id",
            "order_datetime"
        ],
        how="left",
        validate="many_to_one"
    )


def add_merchant_fraud(
    transactions,
    merchant_fraud
):

    fraud = merchant_fraud[
        [
            "merchant_abn",
            "order_datetime",
            "fraud_probability"
        ]
    ].rename(
        columns={
            "fraud_probability":
            "merchant_fraud_probability"
        }
    )

    return transactions.merge(
        fraud,
        on=[
            "merchant_abn",
            "order_datetime"
        ],
        how="left",
        validate="many_to_one"
    )

In [12]:
# ============================================================
# PARTS 2–4 — TRANSACTIONS
# ============================================================

def load_transaction_part(part):

    path = TABLES_DIR / part

    if not path.exists():
        raise FileNotFoundError(
            f"Missing transaction dataset: {path}"
        )

    print(f"Loading {part}...")

    return pd.read_parquet(path)


def clean_transactions(df):

    df = df.copy()

    df["order_datetime"] = pd.to_datetime(
        df["order_datetime"],
        errors="raise"
    ).dt.normalize()

    assert df["order_id"].notna().all()
    assert df["user_id"].notna().all()
    assert df["merchant_abn"].notna().all()

    assert df["order_id"].is_unique, (
        "Duplicate transaction order IDs detected"
    )

    return df

In [13]:
# ============================================================
# MASTER BATCH ETL
# ============================================================

def build_curated_transactions():

    print("=== BNPL BATCH ETL START ===")


    # --------------------------------------------------------
    # 1. LOAD PART 1
    # --------------------------------------------------------

    (
        consumer,
        consumer_details,
        merchants,
        consumer_fraud,
        merchant_fraud
    ) = load_part1()


    # --------------------------------------------------------
    # 2. CLEAN PART 1
    # --------------------------------------------------------

    consumer = clean_consumer(
        consumer
    )

    consumer_fraud = clean_consumer_fraud(
        consumer_fraud
    )

    merchant_fraud = clean_merchant_fraud(
        merchant_fraud
    )


    # --------------------------------------------------------
    # 3. VALIDATE PART 1
    # --------------------------------------------------------

    validate_consumer(
        consumer
    )

    validate_consumer_details(
        consumer_details
    )

    validate_merchants(
        merchants
    )

    validate_consumer_fraud(
        consumer_fraud
    )

    validate_merchant_fraud(
        merchant_fraud
    )

    validate_consumer_relationship(
        consumer,
        consumer_details
    )


    # --------------------------------------------------------
    # 4. PREPARE LOOKUP TABLES
    # --------------------------------------------------------

    consumer_enriched = consumer.merge(
        consumer_details,
        on="consumer_id",
        how="left",
        validate="one_to_one"
    )

    merchants = merchants.reset_index()

    sa2_mapping = load_sa2_mapping()

    seifa = load_seifa()


    # --------------------------------------------------------
    # 5. PROCESS TRANSACTION PARTITIONS SEPARATELY
    # --------------------------------------------------------

    total_rows = 0

    for part in [
        "part_2",
        "part_3",
        "part_4"
    ]:

        output_file = (
            CURATED_DIR
            / f"{part}.parquet"
        )


        # ----------------------------------------------------
        # SKIP SUCCESSFULLY COMPLETED PARTS
        # ----------------------------------------------------

        if output_file.exists():

            print(
                f"\n{part} already exists "
                "— skipping."
            )

            continue


        print("\n" + "=" * 50)
        print(f"PROCESSING {part}")
        print("=" * 50)


        # ----------------------------------------------------
        # LOAD ONE PART
        # ----------------------------------------------------

        transactions = (
            load_transaction_part(part)
        )

        transactions = clean_transactions(
            transactions
        )

        original_rows = len(
            transactions
        )

        print(
            f"{part} transactions loaded: "
            f"{original_rows:,}"
        )


        # ----------------------------------------------------
        # CONSUMER INFORMATION
        # ----------------------------------------------------

        curated = transactions.merge(
            consumer_enriched,
            on="user_id",
            how="left",
            validate="many_to_one"
        )

        del transactions
        gc.collect()


        # ----------------------------------------------------
        # MERCHANT INFORMATION
        # ----------------------------------------------------

        curated = curated.merge(
            merchants,
            on="merchant_abn",
            how="left",
            validate="many_to_one",
            suffixes=(
                "_consumer",
                "_merchant"
            )
        )


        # ----------------------------------------------------
        # CONSUMER FRAUD
        # ----------------------------------------------------

        curated = add_consumer_fraud(
            curated,
            consumer_fraud
        )


        # ----------------------------------------------------
        # MERCHANT FRAUD
        # ----------------------------------------------------

        curated = add_merchant_fraud(
            curated,
            merchant_fraud
        )


        # ----------------------------------------------------
        # FRAUD FLAGS
        # ----------------------------------------------------

        curated["consumer_is_fraud"] = (
            curated[
                "consumer_fraud_probability"
            ]
            >= FRAUD_THRESHOLD
        )

        curated["merchant_is_fraud"] = (
            curated[
                "merchant_fraud_probability"
            ]
            >= FRAUD_THRESHOLD
        )


        # ----------------------------------------------------
        # POSTCODE -> SA2
        # ----------------------------------------------------

        curated["postcode"] = (
            curated["postcode"]
            .astype("string")
            .str.zfill(4)
        )

        curated = curated.merge(
            sa2_mapping[
                [
                    "POA_CODE_2021",
                    "SA2_CODE_2021",
                    "mb_count",
                    "poa_total",
                    "ratio"
                ]
            ],
            left_on="postcode",
            right_on="POA_CODE_2021",
            how="left",
            validate="many_to_one"
        )


        # ----------------------------------------------------
        # SEIFA
        # ----------------------------------------------------

        curated[
            "SA2_CODE_2021"
        ] = pd.to_numeric(
            curated[
                "SA2_CODE_2021"
            ],
            errors="coerce"
        ).astype("Int64")

        curated = curated.merge(
            seifa,
            on="SA2_CODE_2021",
            how="left",
            validate="many_to_one"
        )


        # ----------------------------------------------------
        # FINAL VALIDATION
        # ----------------------------------------------------

        if len(curated) != original_rows:

            raise ValueError(
                f"{part}: transaction "
                "count changed. "
                f"Original={original_rows:,}, "
                f"Final={len(curated):,}"
            )

        assert (
            curated["order_id"].is_unique
        ), (
            f"{part}: duplicate "
            "order IDs detected"
        )


        # ----------------------------------------------------
        # SUMMARY
        # ----------------------------------------------------

        print(
            f"Final transactions: "
            f"{len(curated):,}"
        )

        print(
            "Consumer fraud matched:",
            f"{curated['consumer_fraud_probability'].notna().sum():,}"
        )

        print(
            "Merchant fraud matched:",
            f"{curated['merchant_fraud_probability'].notna().sum():,}"
        )

        print(
            "Missing SA2:",
            f"{curated['SA2_CODE_2021'].isna().sum():,}"
        )

        print(
            "Missing SEIFA:",
            f"{curated['economic_resources_score'].isna().sum():,}"
        )


        # ----------------------------------------------------
        # SAVE THIS PART
        # ----------------------------------------------------

        print(
            f"Saving {part}..."
        )

        curated.to_parquet(
            output_file,
            index=False
        )

        print(
            f"{part} SAVED ✓"
        )

        print(
            output_file
        )

        total_rows += len(curated)


        # ----------------------------------------------------
        # FREE MEMORY
        # ----------------------------------------------------

        del curated
        gc.collect()

        print(
            f"{part} cleared from RAM."
        )


    print(
        "\n=== BNPL BATCH ETL COMPLETE ==="
    )

    print(
        "Curated dataset directory:"
    )

    print(CURATED_DIR)

    return CURATED_DIR

In [14]:
CURATED_DATA_PATH = (
    build_curated_transactions()
)

=== BNPL BATCH ETL START ===


/var/folders/sn/d14740k16x59yrww8r0mdpch0000gn/T/ipykernel_2053/1556919221.py:31: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["order_datetime"] = pd.to_datetime(
/var/folders/sn/d14740k16x59yrww8r0mdpch0000gn/T/ipykernel_2053/1556919221.py:51: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["order_datetime"] = pd.to_datetime(



PROCESSING part_2
Loading part_2...
part_2 transactions loaded: 3,643,266
Final transactions: 3,643,266
Consumer fraud matched: 9,842
Merchant fraud matched: 4
Missing SA2: 600,030
Missing SEIFA: 612,361
Saving part_2...
part_2 SAVED ✓
/Users/amyliu/Desktop/MAST30034_Project_2/data/curated/curated_transactions/part_2.parquet
part_2 cleared from RAM.

PROCESSING part_3
Loading part_3...
part_3 transactions loaded: 4,508,106
Final transactions: 4,508,106
Consumer fraud matched: 70,506
Merchant fraud matched: 4,055
Missing SA2: 743,147
Missing SEIFA: 758,621
Saving part_3...
part_3 SAVED ✓
/Users/amyliu/Desktop/MAST30034_Project_2/data/curated/curated_transactions/part_3.parquet
part_3 cleared from RAM.

PROCESSING part_4
Loading part_4...
part_4 transactions loaded: 6,044,133
Final transactions: 6,044,133
Consumer fraud matched: 0
Merchant fraud matched: 0
Missing SA2: 997,100
Missing SEIFA: 1,017,654
Saving part_4...
part_4 SAVED ✓
/Users/amyliu/Desktop/MAST30034_Project_2/data/curated